In [0]:
!uv pip install -r ../requirements.txt --upgrade

In [0]:
!uv pip install unsloth==2026.7.2 --no-deps

In [0]:
!pip uninstall torchvision -y

In [0]:
%restart_python

In [0]:
%run ../utilities/config

In [0]:
from unsloth import FastLanguageModel
import torch

max_seq_length = DPO_max_seq_length

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=stage2_merged_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [0]:
import json
from datasets import Dataset

rows = []
with open(preference_dataset) as f:
    for line in f:
        rows.append(json.loads(line))

print(f"Loaded {len(rows)} preference pairs")
rows[0]

In [0]:

PROMPT_TEMPLATE = (
    "Below is an instruction from an employee. Write an accurate, "
    "policy-grounded response for the HR Policy Assistant.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n"
)

def format_dpo(ex):
    return {
        "prompt": PROMPT_TEMPLATE.format(instruction=ex["instruction"]),
        "chosen": ex["chosen"] + tokenizer.eos_token,
        "rejected": ex["rejected"] + tokenizer.eos_token,
    }

dpo_dataset = Dataset.from_list(rows).map(format_dpo)
dpo_dataset[0]

In [0]:
dataset = dpo_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = dataset["train"]  
eval_dataset = dataset["test"] 

In [0]:
from trl import DPOConfig, DPOTrainer

dpo_config = DPOConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,             #
    learning_rate=5e-6,             
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    beta=0.1,                       
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=5,
    optim="adamw_8bit",
    output_dir="outputs_dpo",
    report_to="mlflow",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,   # None lets DPOTrainer auto-create a frozen reference from the base
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

In [0]:
import mlflow

# Optional: Set your experiment
mlflow.set_experiment("/Shared/LLM/HRPolicy")

# Enable automatic logging
mlflow.autolog()

# Enable automatic system metrics logging
mlflow.enable_system_metrics_logging()

# Optional: Sample system metrics every 5 seconds
mlflow.set_system_metrics_sampling_interval(5)

with mlflow.start_run(run_name="Stage3_Preference"):

    trainer_stats = dpo_trainer.train()

    print(trainer_stats)
    mlflow.log_param(
        "best_checkpoint",
        dpo_trainer.state.best_model_checkpoint,
    )

    mlflow.log_metric(
        "best_eval_loss",
        dpo_trainer.state.best_metric,
    )

In [0]:
mlflow.autolog(disable=True)

In [0]:
metrics = dpo_trainer.evaluate()
print(metrics)

In [0]:
print(dpo_trainer.state.best_model_checkpoint)
print(dpo_trainer.state.best_metric)

In [0]:
print(dpo_trainer.args.load_best_model_at_end)
print(dpo_trainer.args.metric_for_best_model)
print(dpo_trainer.args.greater_is_better)

In [0]:
from unsloth import FastLanguageModel

best_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=dpo_trainer.state.best_model_checkpoint,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

dpo_trainer.model = best_model

metrics=dpo_trainer.evaluate()
print(metrics)

In [0]:
print("="*60)
print("Best checkpoint :", dpo_trainer.state.best_model_checkpoint)
print("Best eval_loss  :", dpo_trainer.state.best_metric)
print("Current eval    :", metrics["eval_loss"])
print("="*60)

assert abs(metrics["eval_loss"] - dpo_trainer.state.best_metric) < 0.01

In [0]:
import shutil
import tempfile
import os

# Save adapter and tokenizer directly to Volume
best_model.save_pretrained(stage3_adapter_path)
tokenizer.save_pretrained(stage3_adapter_path)
print(f"✓ Adapter saved to {stage3_adapter_path}")

# Ensure merged directory exists
dbutils.fs.mkdirs(stage3_merged_path)

# Save merged model to temp dir first, then copy to Volume
with tempfile.TemporaryDirectory() as temp_dir:
    print(f"Saving merged model to temporary directory: {temp_dir}")
    best_model.save_pretrained_merged(temp_dir, tokenizer, save_method="merged_16bit")
    
    print(f"Copying merged model to Volume: {stage3_merged_path}")
    # Copy contents from temp dir to Volume
    for item in os.listdir(temp_dir):
        src = os.path.join(temp_dir, item)
        dst = os.path.join(stage3_merged_path, item)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"  Copied: {item}")
    
    print("✓ Merged model saved successfully to {}".format(stage3_merged_path))

In [0]:
FastLanguageModel.for_inference(best_model)

def ask(instruction):
    prompt = PROMPT_TEMPLATE.format(instruction=instruction)
    inputs = tokenizer(prompt, return_tensors="pt").to(best_model.device)
    out = best_model.generate(**inputs, max_new_tokens=120, do_sample=False)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip()

print(ask("why health checkup important prior to joining"))
     

In [0]:


print(ask("when I can get medical benefit card? and is there any condition to get it?"))
     

In [0]:
from transformers import AutoModelForCausalLM, AutoTokenizer

merged_model = AutoModelForCausalLM.from_pretrained(
    stage3_merged_path,
    dtype=torch.float16,
)

merged_tokenizer = AutoTokenizer.from_pretrained(stage3_merged_path)
print("load to HF")
merged_model.push_to_hub(preference_merged_repo_name, token=hf_write_token)
merged_tokenizer.push_to_hub(preference_merged_repo_name, token=hf_write_token)